In [137]:
import pandas as pd
import sqlite3

In [138]:
connection = sqlite3.connect("../database/lfpl_oss_household_demographics.db")
cursor = connection.cursor()

In [139]:
connection.execute("DROP TABLE IF EXISTS louisville_zipcodes;")

connection.execute("""
CREATE TABLE louisville_zipcodes (
    zipcode TEXT PRIMARY KEY
);                   
""")
connection.commit()

In [140]:
connection.execute('DROP TABLE IF EXISTS libraries;')

connection.execute('''
CREATE TABLE libraries (
    library_id INT PRIMARY KEY,
    library_name TEXT, 
    latitude REAL,
    longitude REAL,
    zipcode TEXT,
    FOREIGN KEY (zipcode) REFERENCES louisville_zipcodes(zipcode)
);
''')
connection.commit()

In [141]:
connection.execute('DROP TABLE IF EXISTS library_item_details;')

connection.execute('''
CREATE TABLE library_item_details(
    item_id INT PRIMARY KEY,
    title TEXT,
    item_type TEXT,
    item_collection TEXT,
    item_location TEXT,
    item_price INT
);
''')
connection.commit()

In [142]:
connection.execute("DROP TABLE IF EXISTS library_inventory;")

connection.execute("""
CREATE TABLE library_inventory (
    library_id INT NOT NULL,
    item_id INT NOT NULL,
    PRIMARY KEY (library_id, item_id),
    FOREIGN KEY (library_id) REFERENCES libraries(library_id),
    FOREIGN KEY (item_id) REFERENCES library_item_details(item_id)
);
""")

connection.commit()

In [143]:
connection.execute('DROP TABLE IF EXISTS oss_households;')

connection.execute('''
CREATE TABLE oss_households (
    household_id INT PRIMARY KEY,
    date_added TEXT,
    household_type TEXT,
    household_size INT,
    annual_income BIGINT,
    zipcode TEXT,
    FOREIGN KEY (zipcode) REFERENCES louisville_zipcodes(zipcode)
);
''')

connection.commit()

In [144]:
louisville_zipcodes = pd.read_csv('../data/Clean/clean_zips.csv')
louisville_zipcodes.to_sql('louisville_zipcodes', connection, if_exists='append', index=False)

41

In [145]:
libraries = pd.read_csv('../data/Clean/clean_lfpl_loc.csv')
libraries.to_sql('libraries', connection, if_exists='append', index=False)

19

In [146]:
library_item_details = pd.read_csv('../data/Clean/clean_lfpl_inventory.csv')
library_item_details.to_sql('library_item_details', connection, if_exists='append', index=False)

1657554

In [147]:
oss_households = pd.read_csv('../data/Clean/clean_oss.csv')
oss_households.to_sql('oss_households', connection, if_exists='append', index=False)

49613

In [148]:
connection.execute("PRAGMA foreign_keys = ON;")

In [149]:
#Top 10 OSS Household Count per Zipcode and Library
query1 = pd.read_sql('''
SELECT 
    l.library_name,
    COUNT(o.household_id) AS household_count
FROM libraries l
JOIN oss_households o 
    ON l.zipcode = o.zipcode
GROUP BY l.library_name
ORDER BY household_count DESC
LIMIT 10;
''', connection)
query1

,library_name,household_count


In [150]:
test = pd.read_sql('''
SELECT * FROM libraries LIMIT 5;
''', connection)
test

,library_id,library_name,latitude,longitude,zipcode
0,1,BON AIR,38.215179,-85.655178,40220
1,2,FAIRDALE,38.106126,-85.760568,40118
2,3,IROQUOIS,38.182680,-85.771544,40215
3,4,CRESCENT HILL,38.254545,-85.691115,40206
4,5,JEFFERSONTOWN,38.197827,-85.561087,40299


In [151]:
test2 = pd.read_sql('''
SELECT * FROM oss_households LIMIT 5;
''', connection)
test2

,household_id,date_added,household_type,household_size,annual_income,zipcode
0,1,2018-09-11 23:41:00+00:00,Single Person,1,13152.00,40214.0
1,2,2018-09-24 21:28:00+00:00,Single Parent Female,2,0.00,40210.0
2,3,2018-09-24 21:28:00+00:00,Single Parent Female,3,12456.00,40211.0
3,4,2018-09-24 21:30:00+00:00,Single Parent Female,1,10356.00,40299.0
4,5,2018-09-24 21:30:00+00:00,Single Parent Female,2,20672.52,40216.0


In [152]:
#The Sum of OSS Households per Library Branch
query = pd.read_sql ('''

''', connection)
query

TypeError: 'NoneType' object is not iterable

In [ ]:
#Average Annual Income of OSS Households per Library Branch
query = pd.read_sql('''

''', connection)
query

In [ ]:
#Average Amount of Library Items per OSS Household
query = pd.read_sql('''

''', connection)
query